# Mon modèle contre Gemini : qui reconnaît le mieux le thème d'une page ITIE ?

**La question :** au notebook 07b, mon modèle (TF-IDF + régression logistique, entraîné sur le rapport 2022) reconnaît le thème de 93 % des pages du rapport 2023. Est-ce qu'un grand modèle de langage, **Gemini** de Google, fait mieux sur les mêmes pages, sans aucun entraînement ?

**Le protocole :**
- les **mêmes pages** du rapport 2023 ;
- les **mêmes 7 thèmes**, avec la même définition (`data/itie_themes.csv`), envoyée à Gemini dans la consigne ;
- la **même vérité** : le thème donné par le sommaire du rapport ;
- Gemini doit répondre par un des 7 thèmes, rien d'autre, avec `temperature=0` (toujours la même réponse pour la même page).

**Les réponses de Gemini** viennent de `src/gemini_pages.py`, qui enregistre `data/gemini_pages_2023.csv`.

**Une contrainte réelle :** l'offre gratuite de Gemini (modèle `gemini-3.5-flash`) bloque après une vingtaine de pages par jour. On a obtenu **26 pages sur 178** (pages 9 à 35). La comparaison porte donc sur ces 26 pages, et on compare aussi le modèle sur les 178 pages pour situer ce petit échantillon.

In [ ]:
import sys
sys.path.append("..")

import time

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

from src.classifieur_pages import creer_modele
from src.graphiques import graphique_comparaison_gemini

## 1. Le modèle TF-IDF : entraînement sur 2022, prédiction sur 2023

In [ ]:
pages = pd.read_csv("../data/itie/pages_etiquetees.csv")
entrainement = pages[pages["annee"] == 2022]
test = pages[pages["annee"] == 2023]

modele = creer_modele()
debut = time.time()
modele.fit(entrainement["texte"], entrainement["theme"])
duree_entrainement = time.time() - debut

debut = time.time()
test = test.assign(modele_tfidf=modele.predict(test["texte"]))
duree_prediction = time.time() - debut

print(f"Entraînement : {duree_entrainement:.1f} s, prédiction des {len(test)} pages : {duree_prediction:.2f} s")

## 2. Les réponses de Gemini, sur les mêmes pages

In [ ]:
gemini = pd.read_csv("../data/gemini_pages_2023.csv")
comparaison = test.merge(gemini, on="page")                 # on ne garde que les pages où Gemini a répondu
print(f"{len(comparaison)} pages communes, de la page {comparaison['page'].min()} à la page {comparaison['page'].max()}")
comparaison["theme"].value_counts()

**Constat :** les 26 pages couvrent seulement **deux thèmes** sur sept : la synthèse (début du rapport) et le secteur minier. C'est la première limite de cette comparaison : on ne sait pas comment Gemini se débrouille sur les pages de rapprochement, de gouvernance ou d'hydrocarbures.

## 3. Les scores

In [ ]:
plus_frequent = DummyClassifier(strategy="most_frequent").fit(entrainement["texte"], entrainement["theme"])

scores = pd.Series({
    "Modèle TF-IDF (26 pages)": accuracy_score(comparaison["theme"], comparaison["modele_tfidf"]),
    "Gemini 3.5 Flash (26 pages)": accuracy_score(comparaison["theme"], comparaison["theme_gemini"]),
    "Répond toujours le thème le plus fréquent (26 pages)": accuracy_score(comparaison["theme"], plus_frequent.predict(comparaison["texte"])),
    "Modèle TF-IDF (toutes les 178 pages)": accuracy_score(test["theme"], test["modele_tfidf"]),
})
scores.map("{:.0%}".format).to_frame("bonnes réponses")

In [ ]:
graphique_comparaison_gemini(scores, "Sur les mêmes pages, mon petit modèle fait mieux que Gemini")

**Constat :**
- Le modèle TF-IDF : **25 pages sur 26** (96 %). Sur les 178 pages, il reste à 93 % : les 26 pages ne sont pas plus faciles que le reste pour lui.
- Gemini : **13 pages sur 26** (50 %), à peine mieux qu'une réponse fixe « secteur minier » (54 % sur ces pages).

## 4. Où ils ne sont pas d'accord

In [ ]:
desaccords = comparaison[comparaison["modele_tfidf"] != comparaison["theme_gemini"]]
desaccords[["page", "section", "theme", "modele_tfidf", "theme_gemini"]]

In [ ]:
pd.crosstab(comparaison["theme"], comparaison["theme_gemini"], margins=True)

**Constat :** les erreurs de Gemini ne sont pas du hasard, elles ont une logique.
- **Le modèle ne se trompe qu'une fois, page 13** (section 2.3), qu'il classe « revenus » : Gemini fait la même erreur, car cette page de synthèse s'intitule « Revenus générés par le secteur extractif ».
- **Synthèse classée « rapprochement » par Gemini (6 pages, sections 2.4 à 3.4)** : ces pages résument le rapport et décrivent la méthode du rapprochement des paiements. Gemini lit le **sujet** de la page (on y parle de rapprochement), alors que le sommaire la range dans la **partie** « synthèse ».
- **Secteur minier classé « gouvernance » ou « revenus » par Gemini (5 pages)** : ces pages présentent le cadre légal et la fiscalité minière, des sujets qui touchent aussi à la gouvernance et aux revenus.

Le modèle TF-IDF, lui, a appris **la façon dont ce rapport est découpé**, parce qu'il a été entraîné sur le rapport 2022, rédigé par le même cabinet avec le même plan. C'est un avantage réel, mais il faut le dire : la « vérité » choisie (le sommaire) favorise le modèle TF-IDF. Avec une autre vérité (le sujet de chaque page, annoté à la main), l'écart serait sans doute plus petit.

## 5. Le temps et le coût

In [ ]:
temps = pd.DataFrame({
    "modèle TF-IDF": [f"{duree_entrainement:.1f} s", f"{duree_prediction / len(test) * 1000:.1f} ms", "0 €, sur le PC", "aucune"],
    "Gemini 3.5 Flash": ["aucun (déjà entraîné)", f"{gemini['secondes'].median():.0f} s (médiane)", "gratuit, mais ~20 pages par jour", "le texte part chez Google"],
}, index=["entraînement", "temps par page", "coût", "données envoyées à l'extérieur"])
temps

**Constat :**
- Le modèle classe une page en **moins d'une milliseconde**. Gemini met environ **10 secondes** par page (médiane), et parfois plus d'une minute quand il faut attendre la limite d'appels.
- Pour les 178 pages, il met moins d'une seconde. Gemini aurait demandé **plus d'une semaine** avec l'offre gratuite.
- Il tourne **sur le PC** : aucun texte n'est envoyé à une entreprise extérieure, ce qui compte pour des documents confidentiels.

## Conclusions

1. Sur une tâche précise, avec des exemples pour apprendre, **un petit modèle entraîné bat un grand modèle généraliste** : 96 % contre 50 % sur les mêmes 26 pages.
2. Il est **des milliers de fois plus rapide**, gratuit et ne fait sortir aucune donnée.
3. Gemini n'a pas « tort » au hasard : il classe une page selon son **sujet**, le modèle TF-IDF selon **la partie du rapport**. Ce n'est pas la même question, et c'est la définition du sommaire qui sert de référence ici.

## Limites

- **26 pages seulement**, et deux thèmes sur sept : on ne peut pas généraliser ce score de 50 %. Une différence de 12 pages sur 26 reste pourtant trop grande pour être due au hasard.
- Un seul modèle Gemini, avec une seule consigne. Une consigne plus détaillée, ou quelques exemples de pages déjà classées dans la consigne, l'aideraient probablement.
- Gemini a un vrai avantage que ce test ne mesure pas : il **comprend les phrases** et peut répondre à une question précise (« combien SMB a-t-elle payé en 2023 ? »). Le modèle TF-IDF ne sait que ranger des pages.

**Suite (étape 8) :** combiner les deux. Le modèle TF-IDF trouve vite les bonnes pages, puis un modèle de langage lit seulement ces pages pour répondre à une question : c'est le principe du **RAG** (*retrieval-augmented generation*).